# 03 · Auto Schema × schema explícito

O Weaviate cria a collection e as propriedades sozinho quando você insere um objeto que ele não conhece: é o **Auto Schema**. Ótimo para prototipar, perigoso em produção:

- o **tipo** é inferido do primeiro valor que aparece;
- **toda** propriedade de texto ganha índice de filtro **e** índice de busca (BM25);
- um erro de digitação vira uma propriedade nova, em silêncio.

In [1]:
import subprocess
import time

import weaviate
from weaviate.classes.config import Configure, DataType, Property, Tokenization
import utils

client = weaviate.connect_to_local()

/Users/dudanogueira/dev/weaviate/lab/claude/latinoware-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Deixando o Auto Schema decidir

In [2]:
client.collections.delete("Produto")
produtos = client.collections.use("Produto")  # ainda não existe!

produtos.data.insert({
    "nome": "Camiseta Latinoware 2026",
    "preco": 59,
    "estoque": 120,
    "lancamento": "2026-10-14T09:00:00Z",
    "tags": ["evento", "algodão"],
    "ativo": True,
    "sku": "LTW-2026-001",
})

for p in produtos.config.get().properties:
    print(f"{p.name:11} {p.data_type.value:9} filtro={p.index_filterable!s:5} busca={p.index_searchable!s:5} tokenização={p.tokenization.value if p.tokenization else '-'}")

ativo       boolean   filtro=True  busca=False tokenização=-
sku         text      filtro=True  busca=True  tokenização=word
nome        text      filtro=True  busca=True  tokenização=word
preco       number    filtro=True  busca=False tokenização=-
estoque     number    filtro=True  busca=False tokenização=-
lancamento  date      filtro=True  busca=False tokenização=-
tags        text[]    filtro=True  busca=True  tokenização=word


Repare:

- `preco` e `estoque` viraram `number` (float), não `int`;
- a data foi reconhecida, mas só porque estava em RFC 3339; outro formato vira `text`;
- `sku` ganhou tokenização `word`: `LTW-2026-001` vira os tokens `ltw`, `2026`, `001`, e um filtro por `2026` acha todos os SKUs que têm esse pedaço.

## 2. O primeiro valor manda

In [3]:
try:
    produtos.data.insert({"nome": "Caneca", "estoque": "dez unidades"})
except Exception as e:
    print("Erro:", str(e)[:200])

Erro: Object was not added! Unexpected status code: 422, with response body: {'error': [{'message': "invalid object: invalid number property 'estoque' on class 'Produto': requires a float, the given value i


## 3. Um nome diferente vira schema

Com acento nem entra (nomes de propriedade seguem a regra do GraphQL):

In [4]:
try:
    produtos.data.insert({"nome": "Boné", "preço": 39})
except Exception as e:
    print("Erro:", str(e)[-170:])

Erro: te collection: 'preço' is not a valid property name. Property names in Weaviate are restricted to valid GraphQL names, which must be “/[_A-Za-z][_0-9A-Za-z]{0,230}/”"}]}.


E se outro serviço (ou outro dev) escrever em inglês?

In [5]:
produtos.data.insert({"nome": "Boné", "price": 39})
print([p.name for p in produtos.config.get().properties])

['ativo', 'sku', 'nome', 'preco', 'estoque', 'lancamento', 'tags', 'price']


Agora existem `preco` **e** `price`, e metade dos filtros vai dar resultado errado. Em produção: **`AUTOSCHEMA_ENABLED=false`** e schema versionado junto com o código.

## 4. Quanto custam os índices que você não usa?

Os mesmos 25 mil artigos, duas vezes:

| Propriedade | Auto Schema | Explícito |
|---|---|---|
| `title` | filtro + busca | filtro + busca |
| `text` | filtro + busca | só busca (ninguém filtra por um parágrafo inteiro) |
| `url` | filtro + busca | nenhum índice: só é exibida |
| `wiki_id` | filtro + busca, tokenização `word` | só filtro, tokenização `field` (valor exato) |

In [6]:
dados = utils.load_wiki(1)


def importar(colecao):
    inicio = time.perf_counter()
    with colecao.batch.fixed_size(batch_size=500) as batch:
        for linha in dados.itertuples():
            batch.add_object(
                {"title": linha.title, "text": linha.text, "url": linha.url, "wiki_id": linha.wiki_id},
                vector=linha.vector,
            )
    assert not colecao.batch.failed_objects
    utils.wait_for_indexing(client, colecao.name)
    return time.perf_counter() - inicio


client.collections.delete(["WikiAuto", "WikiExplicita"])

tempo_auto = importar(client.collections.use("WikiAuto"))

explicita = client.collections.create(
    "WikiExplicita",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT, index_filterable=False),
        Property(name="url", data_type=DataType.TEXT, index_filterable=False, index_searchable=False),
        Property(name="wiki_id", data_type=DataType.TEXT, tokenization=Tokenization.FIELD, index_searchable=False),
    ],
    vector_config=Configure.Vectors.self_provided(),
)
tempo_explicito = importar(explicita)

print(f"import Auto Schema: {tempo_auto:.1f}s | explícito: {tempo_explicito:.1f}s")

import Auto Schema: 6.4s | explícito: 5.8s


Cada índice é um **bucket** no disco (uma árvore LSM). A collection tem 3 shards, um em cada nó; vamos olhar dentro dos containers (só para a demo; em produção isso aparece nas métricas):

In [7]:
NOS = ["latinoware-cluster-weaviate-0-1", "latinoware-cluster-weaviate-1-1", "latinoware-cluster-weaviate-2-1"]


def no_container(no, comando):
    return subprocess.run(["docker", "exec", no, "sh", "-c", comando], capture_output=True, text=True).stdout


for nome in ["wikiauto", "wikiexplicita"]:
    buckets = sorted(set(no_container(NOS[0], f"ls /var/lib/weaviate/{nome}/*/lsm").split()))
    kb = sum(int(x) for no in NOS for x in no_container(no, f"du -sk /var/lib/weaviate/{nome}/*/lsm | cut -f1").split())
    print(f"{nome:14} {len(buckets)} buckets por shard, {kb / 1024:.0f} MB em lsm/ (3 nós)")
    print("   ", [b for b in buckets if b.startswith("property_") and not b.startswith("property__")])

wikiauto       11 buckets por shard, 233 MB em lsm/ (3 nós)
    ['property_text', 'property_text_searchable', 'property_title', 'property_title_searchable', 'property_url', 'property_url_searchable', 'property_wiki_id', 'property_wiki_id_searchable']


wikiexplicita  7 buckets por shard, 171 MB em lsm/ (3 nós)
    ['property_text_searchable', 'property_title', 'property_title_searchable', 'property_wiki_id']


Menos índices = menos disco, menos memória, import e compactação mais rápidos. Com 25 mil objetos a diferença é pequena; com 100 milhões, é a diferença entre caber ou não caber no nó.

**Regra prática:** para cada propriedade, pergunte

1. vou **filtrar** por ela? → `index_filterable`
2. vou fazer **BM25/híbrida** nela? → `index_searchable` (e qual tokenização?)
3. vou filtrar por **faixa** (`>`, `<`) em número/data? → `index_range_filters`
4. nada disso? → desligue tudo, ela só é armazenada.

➡️ `04_indices_vetoriais.ipynb`

In [8]:
client.close()